In [1]:
import pandas as pd
import tarfile
import xml.etree.ElementTree as ET
from xml.dom.minidom import parse, parseString
from collections import Counter
import numpy as np
import re
import csv
from tqdm import tqdm
import csv
import pickle
import matplotlib.pyplot as plt
import time
import os
import copy
from scipy.stats import anderson
import statsmodels.api as sm
from datetime import datetime
import math
from ordered_set import OrderedSet
import category_encoders as ce
from sklearn.preprocessing import RobustScaler
import pickle

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
path = r"C:\Users\bbinnend\OneDrive\Thesis\open_data_can_2020_10.07.23.csv"
df = pd.read_csv(path)

In [ ]:
df.head()

In [5]:
keep_cols = ["ID_NOTICE_CAN", "CANCELLED", "ISO_COUNTRY_CODE", "CAE_TYPE", "MAIN_ACTIVITY", "B_ON_BEHALF", "B_INVOLES_JOINT_PROCUREMENT",
             "B_AWARDED_BY_CENTRAL_BODY", "TYPE_OF_CONTRACT", "B_FRA_AGREEMENT", "FRA_ESTIMATED", "CPV",
             "B_EU_FUNDS", "B_RENEWALS", "NUMBER_TENDERS_SME", 
             "TOP_TYPE", "CRIT_CODE", "CRIT_PRICE_WEIGHT", "B_RECURRENT_PROCUREMENT", "INFO_ON_NON_AWARD",
             "NUMBER_OFFERS", "AWARD_VALUE_EURO_FIN_1", "DT_AWARD", "LOTS_NUMBER", "ID_LOT"] #ID_NOTICE_CAN

In [6]:
df = df.drop(columns = [col for col in df.columns if col not in keep_cols])

--------------------------------------------------------
DROP ROWS
--------------------------------------------------------
--------------------------------------------------------

In [7]:
#remove all canceled tenders
missing_cancelled_tenders = list(df.loc[df["CANCELLED"] == 1].index.values)
#missing rows for B_AWARDED_BY_CENTRAL_BODY
missing_awarded_by_central_body = list(df.loc[pd.isna(df["B_AWARDED_BY_CENTRAL_BODY"]) == True].index.values)
#missing rows for procedure type
missing_procedure = list(df.loc[pd.isna(df["TOP_TYPE"]) == True].index.values)
#missing rows for B_EU_FUNDS
missing_eu_fund = list(df.loc[pd.isna(df["B_EU_FUNDS"]) == True].index.values)
#missing rows for CRIT_CODE
missing_crit_code = list(df.loc[pd.isna(df["CRIT_CODE"]) == True].index.values)
#rows for non awarded contract
missing_non_awarded_contracts = list(df.loc[pd.isna(df["INFO_ON_NON_AWARD"]) == False].index.values)
#remove missing contract award values
missing_award_value = list(df.loc[pd.isna(df["AWARD_VALUE_EURO_FIN_1"]) == True].index.values)

#combine all lists
drop_rows_indices = (missing_award_value + missing_awarded_by_central_body + missing_crit_code + 
                     missing_eu_fund + missing_non_awarded_contracts + missing_procedure + missing_cancelled_tenders)

#drop selection of rows
df = df.drop(labels = drop_rows_indices, axis = 0).reset_index(drop=True)

--------------------------------------------------------
CLEAN FRAMEWORK AGREEMENT COLS
--------------------------------------------------------
--------------------------------------------------------

we distinguish three cases: <br>
(1) a framework was indicated --> do nothing <br>
(2) no framework was indicated, but one of the indicators is present --> assign framework <br>
(3) value is absent, but one of the indicators is present --> assign framework

In [8]:
#in case of A/KA/AC/KAC and C/KC we assume a framework agreement was used
#define the indicators
indicators = ["A", "KA", "AC", "KAC", "C", "KC"]
#find all cases of instance (2)
indices_assign_framework_case_2 = list(df.loc[(df["B_FRA_AGREEMENT"] == "N") & (
    (df["FRA_ESTIMATED"] == indicators[0]) |
    (df["FRA_ESTIMATED"] == indicators[1]) |
    (df["FRA_ESTIMATED"] == indicators[2]) |
    (df["FRA_ESTIMATED"] == indicators[3]) |
    (df["FRA_ESTIMATED"] == indicators[4]) |
    (df["FRA_ESTIMATED"] == indicators[5])
)].index.values)

#find all cases of instance (3)
indices_assign_framework_case_3 = list(df.loc[(pd.isna(df["B_FRA_AGREEMENT"]) == True) & (
    (df["FRA_ESTIMATED"] == indicators[0]) |
    (df["FRA_ESTIMATED"] == indicators[1]) |
    (df["FRA_ESTIMATED"] == indicators[2]) |
    (df["FRA_ESTIMATED"] == indicators[3]) |
    (df["FRA_ESTIMATED"] == indicators[4]) |
    (df["FRA_ESTIMATED"] == indicators[5])
)].index.values)

#combine the lists
indices_assign_framework = indices_assign_framework_case_2 + indices_assign_framework_case_3

#assign framework agreement for those rows in case 2 or 3
for indice in indices_assign_framework:
    df.at[indice, "B_FRA_AGREEMENT"] == "Y"

print(len(indices_assign_framework_case_2), len(indices_assign_framework_case_3))

65404 0


--------------------------------------------------------
PREPROCESS LOTS COLUMNS
--------------------------------------------------------
--------------------------------------------------------

In [9]:
#MAKE NEW COLUMN, SCORE 1 IF DIVIDED IN LOTS, STATE 0 IF NOT, ASSUME NOT IF NO LOTS LISTED
lots_col = []

for i in range(0, len(df)):
    if pd.isna(df["LOTS_NUMBER"].iloc[i]) == True:
        lots_col.append("N")
    elif df["LOTS_NUMBER"].iloc[i] == 1:
        lots_col.append("N")
    elif df["LOTS_NUMBER"].iloc[i] > 1:
        lots_col.append("Y")
    else:
        continue

df["lots"] = lots_col

--------------------------------------------------------
PREPROCESS NUMERS OF OFFERS AND SME OFFERS
--------------------------------------------------------
--------------------------------------------------------

In [ ]:
pd.isna(df).sum()

In [ ]:
#IF NUMBER_OFFERS > 1 AND NUMBER_TENDERS_SME IS NaN, ASSUME IT IS 0
number_tenders_sme_col = []

for i in tqdm(range(0, len(df))):
    if pd.isna(df["NUMBER_OFFERS"].iloc[i]) == False and pd.isna(df["NUMBER_TENDERS_SME"].iloc[i]) == True:
        number_tenders_sme_col.append(0)
    elif pd.isna(df["NUMBER_TENDERS_SME"].iloc[i]) == False:
        number_tenders_sme_col.append(df["NUMBER_TENDERS_SME"].iloc[i])
    else:
        number_tenders_sme_col.append(np.nan)

df["NUMBER_TENDERS_SME"] = number_tenders_sme_col

--------------------------------------------------------
PREPROCESS CPV CODES
--------------------------------------------------------
--------------------------------------------------------

In [12]:
#transform cpv column to 2 letter acronyms
def transform_cpv(df, classification_type = 2):
    values = []
    for i in tqdm(range(0, len(df))):
        value = df["CPV"].iloc[i]
        if pd.isna(value) == False:
            values.append(str(value)[:classification_type])
        else:
            values.append(np.nan)
            print("this should not be happenening")
    df["CPV"] = values
    return df

In [ ]:
df = transform_cpv(df)

--------------------------------------------------------
DROP COLUMNS
--------------------------------------------------------
--------------------------------------------------------

In [14]:
#define list of columns that need to be dropped
drop_cols = ["CANCELLED", "FRA_ESTIMATED", "INFO_ON_NON_AWARD", "DT_AWARD"]
df = df.drop(columns = drop_cols)

In [ ]:
#CHECK COLUMNS WITH MISSING VALUES
pd.isna(df).sum()

In [15]:
#DROP THE ROWS WHERE NUMER OF OFFERS ARE MISSING
missing_number_offers = list(df.loc[pd.isna(df["NUMBER_OFFERS"]) == True].index.values)
df = df.drop(labels = missing_number_offers).reset_index(drop=True)

In [ ]:
#let's assess the price criteria for a given award criteria. 334008 have L and 82415 have M
df["CRIT_CODE"].loc[pd.isna(df["CRIT_PRICE_WEIGHT"]) == True].value_counts()

#set the CRIT_PRICE_WEIGHT to 100 when criteria is L
crit_price_weight_col = []
drop_empty_price_weight = []

for i in tqdm(range(0, len(df))):
    if pd.isna(df["CRIT_PRICE_WEIGHT"].iloc[i]) == True and df["CRIT_CODE"].iloc[i] == "L":
        crit_price_weight_col.append(100)
    elif pd.isna(df["CRIT_PRICE_WEIGHT"].iloc[i]) == False:
        crit_price_weight_col.append(df["CRIT_PRICE_WEIGHT"].iloc[i])
    elif pd.isna(df["CRIT_PRICE_WEIGHT"].iloc[i]) == True and df["CRIT_CODE"].iloc[i] == "M":
        crit_price_weight_col.append(np.nan)
        drop_empty_price_weight.append(i)
    else:
        continue

#REPLACE COLUMN WITH NEW VALUES AND DROP MISSING ONES
df["CRIT_PRICE_WEIGHT"] = crit_price_weight_col
df = df.drop(labels = drop_empty_price_weight).reset_index(drop=True)

In [17]:
def preprocess_weight_columns(text: str):
    count = 0
    text = str(text)
    pattern = r'\d+(?:[,.]\d+)?'
    numbers_list = re.findall(pattern, text)
    if len(numbers_list) > 0:
        for value in numbers_list:
            #the value has a comma, replace and multiply
            if "," in value:
                value = float(value.replace(",", "."))
                if value <= 1.0:
                    value = int(float(value) * 100)
                else:
                    value = int(value)
            elif "." in value and float(value) <= 1.0:
                value = int(float(value) * 100)
            elif "." in value and float(value) > 1.0:
                value = int(float(value))
            else:
                value = int(float(value))
    else:
        value = np.nan
    #print("{} were set to 100 because > 100".format(count))
    return value

In [ ]:
crit_price_clean_col = []
count = 0

for i in tqdm(range(0, len(df))):
    value = preprocess_weight_columns(df["CRIT_PRICE_WEIGHT"].iloc[i])
    crit_price_clean_col.append(value)

print(len(df) == len(crit_price_clean_col))
df["CRIT_PRICE_WEIGHT"] = crit_price_clean_col

--------------------------------------------------------
MAKE DATASET
--------------------------------------------------------
--------------------------------------------------------

In [19]:
def filter_outliers(df, upper, lower, target_col = "AWARD_VALUE_EURO_FIN_1"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers, with high amount = {}, and low amount = {}".format(len(outlier_indices), high_amount, low_amount))
    df = df.drop(labels = outlier_indices, axis = 0)
    return df

In [6]:
def train_validate_test_split(df, train, validate):
    target_col = "AWARD_VALUE_EURO_FIN_1"
    #df = df.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
    train_indice = int(train * len(df))
    validate_indice = int(validate * len(df))
    test_indice = int((1-validate-train) * len(df))

    train_set = df[:train_indice]
    val_set = df[train_indice:validate_indice]
    test_set = df[validate_indice: test_indice]

    X_train = train_set.drop(columns = [target_col]).values
    y_train = train_set[target_col].values

    X_val = val_set.drop(columns = [target_col]).values
    y_val = val_set[target_col].values

    X_test = test_set.drop(columns = [target_col]).values
    y_test = test_set[target_col].values

    return X_train, y_train, X_val, y_val, X_test, y_test

In [7]:
def encode_data(df, target_column="award_contract/val_total"):
    
    base_n_encoder_cols = ["ISO_COUNTRY_CODE", "MAIN_ACTIVITY", "CPV"]
    one_hot_cols = ["CAE_TYPE", "B_ON_BEHALF", "B_AWARDED_BY_CENTRAL_BODY", "TYPE_OF_CONTRACT",
                    "B_FRA_AGREEMENT", "B_EU_FUNDS", "TOP_TYPE", "CRIT_CODE", "lots"]

    encoder = ce.BaseNEncoder(cols=base_n_encoder_cols, return_df=True, base=2)
    df = encoder.fit_transform(df)
    df = pd.get_dummies(df, columns=(one_hot_cols), drop_first=True, dtype=int)

    return df

In [8]:
def scale_data(df):
    numerical_cols = ["LOTS_NUMBER", "CRIT_PRICE_WEIGHT", "NUMBER_OFFERS", "NUMBER_TENDERS_SME"]
    scaler = RobustScaler()

    numeric_data = df[numerical_cols].values.reshape((len(df), len(numerical_cols)))
    scaled_num_data = scaler.fit_transform(numeric_data)
    df_numeric = pd.DataFrame(data=scaled_num_data,columns = numerical_cols)
    df[numerical_cols] = df_numeric

    return df

In [23]:
df_test = copy.deepcopy(df)

In [ ]:
df_test = filter_outliers(df_test, upper = 0.90, lower = 0.35)

In [ ]:
len(df)

In [25]:
#with open("new_data/df_from_csv_preprocessed.pickle", "wb") as f:
#    pickle.dump(df, f)

In [117]:
df = pd.read_pickle("new_data/df_merged_dataset_clean").reset_index(drop=True)

In [118]:
drop_cols=["ID_NOTICE_CAN", "short description"]
df = df.drop(columns = drop_cols)

In [ ]:
pd.isna(df).sum()

In [120]:
crit_price_weight_empty = list(df.loc[pd.isna(df["CRIT_PRICE_WEIGHT"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

crit_price_weight_empty = list(df.loc[pd.isna(df["NUMBER_OFFERS"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

crit_price_weight_empty = list(df.loc[pd.isna(df["NUMBER_TENDERS_SME"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)


In [ ]:
df.head()

In [ ]:
#SCALE DATA
df = filter_outliers(df, upper=0.90, lower = 0.20)

In [123]:
# Initialize the RobustScaler
robust_scaler = RobustScaler()

# Fit and transform the DataFrame
df[["LOTS_NUMBER", "CRIT_PRICE_WEIGHT", "NUMBER_OFFERS", "NUMBER_TENDERS_SME"]] = pd.DataFrame(robust_scaler.fit_transform(df[["LOTS_NUMBER", "CRIT_PRICE_WEIGHT", "NUMBER_OFFERS", "NUMBER_TENDERS_SME"]]))
#ENCODE DATA
df = encode_data(df)

In [ ]:
len(df)

In [ ]:
pd.isna(df).sum()

In [126]:
crit_price_weight_empty = list(df.loc[pd.isna(df["CRIT_PRICE_WEIGHT"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

crit_price_weight_empty = list(df.loc[pd.isna(df["NUMBER_OFFERS"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

crit_price_weight_empty = list(df.loc[pd.isna(df["NUMBER_TENDERS_SME"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

crit_price_weight_empty = list(df.loc[pd.isna(df["LOTS_NUMBER"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

In [ ]:
len(df)

In [129]:
target_column = "AWARD_VALUE_EURO_FIN_1"
X = df.drop(columns = target_column).values
y = df[target_column].values
#SPLIT DATASET
#X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df, 0.6, 0.2)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
len(X_train)

In [132]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Create a RandomForestRegressor
rf_regressor = RandomForestRegressor()

# Define the parameter grid for GridSearch
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Create GridSearchCV object
grid_search = GridSearchCV(estimator=rf_regressor, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the model with GridSearch
grid_search.fit(X_train, y_train)

# Get the best parameters from the grid search
best_params = grid_search.best_params_
print("Best Parameters:", best_params)

# Get the best model from the grid search
best_rf_regressor = grid_search.best_estimator_

# Predict on the test set
y_pred = best_rf_regressor.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("R-squared:", r2)

In [17]:
with open("Data/final_data_set_from_csv.pickle", "rb") as f:
    df = pickle.load(f)

In [18]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM
import matplotlib.pyplot as plt
#from keras.utils import plot_model
import os
import torch
from collections import Counter
import tensorflow as tf
from tensorflow.keras import optimizers
from tensorflow.keras import metrics
from tensorflow.keras import losses
from transformers import pipeline
from keras.layers import Softmax
from tensorflow.keras.utils import plot_model
from tensorflow.keras import regularizers
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import category_encoders as ce

In [19]:
def create_model(input_dimension, X_train, y_train, X_val, y_val, X_test, y_test, learning_rate = 0.001, epochs = 50):
    #let's play around a little more with the keras model
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate) #tf.keras.optimizers.experimental.Adagrad(learning_rate=0.001)
    loss_function = "mae"
    metrics = ["mae", "mse"]

    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(32, activation = "relu")(input_num_cat) #kernel_regularizer=regularizers.L1L2(l1=0.005)
    x = Dropout(rate = 0.2)(x)
    x = Dense(8, activation = "relu")(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(32, activation = "relu")(x)
    regression_layer = Dense(1, activation="linear")(x)
    model_num_cat = Model(inputs = [input_num_cat],
                          outputs = regression_layer)

    model_num_cat.compile(loss=loss_function,
                          optimizer = optimizer,
                          metrics = metrics)
    model_num_cat.summary()

    history = model_num_cat.fit(x = [X_train], y=y_train,
                                  validation_data = (X_val, y_val),
                                  epochs = epochs,
                                  batch_size = 32)

    y_pred = model_num_cat.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    
    metric = tf.keras.metrics.R2Score()
    metric.update_state(y_test.reshape(-1,1), y_pred)
    r2_result = metric.result()
    r2_result.numpy()

    return history, mae, r2_result

In [ ]:
df.head()

In [22]:
#SPLIT DATASET
X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df, 0.6, 0.2)

In [ ]:
#run on the entire num_cat dataset
history, mae, r2_result = create_model(input_dimension=X_train.shape[1], 
             X_train = X_train, 
             y_train = y_train, 
             X_val = X_val, 
             y_val = y_val, 
             X_test = X_test, 
             y_test = y_test,
             learning_rate= 0.001,
             epochs = 50)